# Praktikum Logic Agents: Agen Petugas Yudisium (`FolKB`)

**Skenario.** Kamu adalah agen di loket yudisium UnivImpian. Data mahasiswa (Joko, Wowo, Sorya) datang bertahap. Di setiap tahap, tentukan apa yang **sudah terbukti** dan apa yang **belum**.

Deskripsi soal, kosakata predikat, dan aturan R1 sampai R9 ada di dokumen soal.

**Aturan main**
1. `FolKB` hanya mendukung definite clause, jadi tidak ada `~`. Fakta "tidak DO" ditulis sebagai `TidakDO(m)`.
2. Huruf kapital = konstanta (`Wowo`, `Smt9`, `UnivImpian`), huruf kecil = variabel.
3. Hanya predikat **dasar** yang boleh di-`tell`. Predikat turunan harus dibuktikan oleh aturan, bukan di-`tell`.
4. Jangan pakai `if fol_bc_ask(...)` langsung. Gunakan helper `terbukti()`.

Pastikan file `logic.py` (aima-python) berada satu folder dengan notebook ini. Kerjakan setiap sel yang bertanda `TODO`.

In [ ]:
import copy
from logic import FolKB, expr, fol_fc_ask, fol_bc_ask

K = "UnivImpian"

def terbukti(kb, query: str) -> bool:
    """True jika query (tanpa variabel) dapat dibuktikan dari kb."""
    return next(fol_bc_ask(kb, expr(query)), None) is not None

def tell_semua(kb, fakta):
    """Contoh: tell_semua(kb, ["TidakDO(Wowo)", "LunasUKT(Wowo)"])"""
    for f in fakta:
        kb.tell(expr(f))

## Tahap 0: Bangun Knowledge Base
Terjemahkan R1 sampai R9 dari dokumen soal menjadi klausa `expr(...)`, lalu masukkan juga fakta durasi studi valid.

In [ ]:
def build_akademik_kb() -> FolKB:
    clauses = []
    # TODO 0.1: tulis R1 sampai R9 sebagai expr("... ==> ...") dan masukkan ke clauses
    ...

    # TODO 0.2: tulis fakta DurasiStudiValid untuk Smt7 sampai Smt14 dan masukkan ke clauses
    ...
    return FolKB(clauses)

Pengecek mandiri untuk klausa (sel ini sudah diberikan, jangan diubah):

In [ ]:
def cek_kb():
    """Pengecek mandiri (diberikan). Jalankan untuk memeriksa klausamu."""
    def lengkap(m, smt):
        return [f"TerdaftarKampus({m}, {K})", f"LamaStudi({m}, {smt})", f"TidakDO({m})",
                f"SKSMinimalTerpenuhi({m})", f"IPKMemenuhi({m})", f"BebasNilaiE({m})",
                f"LulusSidangSkripsi({m})", f"RevisiDisahkan({m})",
                f"LunasUKT({m})", f"DataPDDIKTIValid({m})"]
    tanpa = lambda kata: [f for f in lengkap("X", "Smt8") if kata not in f]
    kasus = [
        ("Semua lengkap, Smt8",    lengkap("X", "Smt8"),  True),
        ("Semua lengkap, Smt14",   lengkap("X", "Smt14"), True),
        ("Semua lengkap, Smt15",   lengkap("X", "Smt15"), False),
        ("Tanpa LunasUKT",         tanpa("LunasUKT"),     False),
        ("Tanpa DataPDDIKTIValid", tanpa("PDDIKTI"),      False),
        ("Tanpa TidakDO",          tanpa("TidakDO"),      False),
        ("Tanpa RevisiDisahkan",   tanpa("Revisi"),       False),
        ("Tanpa BebasNilaiE",      tanpa("BebasNilaiE"),  False),
    ]
    for nama, fakta, harapan in kasus:
        kb = build_akademik_kb()
        tell_semua(kb, fakta)
        hasil = terbukti(kb, "Ijazah(X)")
        print(f"[{'OK' if hasil == harapan else 'SALAH'}] {nama:<24} Ijazah(X) = {hasil} (seharusnya {harapan})")

cek_kb()

## Tahap 1: Wowo baru masuk loket
Fakta: terdaftar di `UnivImpian`, lama studi `Smt9`, `TidakDO`.

In [ ]:
kb_wowo = build_akademik_kb()

# TODO 1.1: tell fakta dasar Wowo ke kb_wowo
...

# TODO 1.2: cetak hasil terbukti() untuk:
#   MasaStudiNormal(Wowo)
#   MahasiswaSah(Wowo, UnivImpian)
#   Ijazah(Wowo)
...

## Tahap 2: Berkas akademik Wowo masuk
Tambahkan: `SKSMinimalTerpenuhi`, `IPKMemenuhi`, `BebasNilaiE`, `LulusSidangSkripsi`, `RevisiDisahkan`.

In [ ]:
# TODO 2.1: tell kelima fakta akademik Wowo
...

# TODO 2.2: cetak hasil terbukti() untuk:
#   LulusYudisium(Wowo, UnivImpian)
#   Ijazah(Wowo)
...

## Tahap 3: Diagnosis
`Ijazah(Wowo)` masih gagal. Apa yang kurang?

In [ ]:
# TODO 3.1: telusuri mundur dari R9, lalu isi daftar fakta minimal yang hilang
fakta_hilang = []   # contoh isi: ["NamaPredikat(Wowo)", ...]
print("Fakta yang hilang:", fakta_hilang)

# TODO 3.2: Wowo HANYA melunasi UKT. tell LunasUKT(Wowo), lalu cetak
#   terbukti() untuk BebasTanggungan(Wowo) dan Ijazah(Wowo)
...

# TODO 3.3: tell fakta terakhir yang kamu temukan, lalu cetak terbukti() untuk Ijazah(Wowo)
...

## Tahap 4: Kasus Sorya (Smt15)
Sorya punya semua fakta dasar, akademik, dan administrasi yang lengkap, tetapi `LamaStudi(Sorya, Smt15)`.

In [ ]:
kb_sorya = build_akademik_kb()

# TODO 4.1: tell semua fakta Sorya (dasar + akademik + administrasi),
#   lalu cetak terbukti() untuk MasaStudiNormal(Sorya) dan Ijazah(Sorya)
...

# TODO 4.2 (eksperimen): salin KB dengan copy.deepcopy(kb_sorya), tell DurasiStudiValid(Smt15)
#   pada salinan itu, lalu cetak terbukti() untuk Ijazah(Sorya)
...

## Tahap 5: Semua mahasiswa dan query umum (FC vs BC)
- **Joko**: `Smt8`, semua syarat lengkap
- **Wowo**: `Smt9`, semua lengkap kecuali `LunasUKT`
- **Sorya**: `Smt15`, semua lengkap

In [ ]:
def data_semua_mahasiswa() -> list:
    """TODO 5.1: kembalikan daftar string fakta DASAR untuk Joko, Wowo, dan Sorya.
    Ingat: jangan memasukkan predikat turunan."""
    fakta = []
    # ...
    return fakta

In [ ]:
def uji_forward_chaining(kb_fc: FolKB):
    jumlah_awal = len(kb_fc.clauses)
    # TODO 5.2: jalankan fol_fc_ask(kb_fc, expr("Ijazah(m)")), ubah hasilnya ke list,
    #   lalu kumpulkan nama penerima ke set (petunjuk: s[expr("m")], ubah ke str).
    penerima = set()
    # ...

    # TODO 5.3: cetak seluruh fakta baru: kb_fc.clauses[jumlah_awal:]
    ...
    return penerima


def uji_backward_chaining(kb_bc: FolKB):
    # TODO 5.4: untuk Joko, Wowo, Sorya, cetak apakah Ijazah(nama) terbukti
    ...

    # TODO 5.5: jalankan fol_bc_ask(kb_bc, expr("Ijazah(m)")), kumpulkan penerima ke set
    penerima = set()
    # ...
    return penerima

In [ ]:
data = data_semua_mahasiswa()
kb_fc, kb_bc = build_akademik_kb(), build_akademik_kb()   # dua KB TERPISAH
tell_semua(kb_fc, data)
tell_semua(kb_bc, data)

penerima_fc = uji_forward_chaining(kb_fc)
penerima_bc = uji_backward_chaining(kb_bc)

# TODO 5.6: cetak apakah penerima_fc == penerima_bc
...

# TODO 5.7: di kb_bc, cetak terbukti() untuk jejak kegagalan:
#   LulusYudisium(Wowo, UnivImpian), BerkasIjazahLengkap(Wowo, UnivImpian),
#   MasaStudiNormal(Sorya), LulusYudisium(Sorya, UnivImpian)
...

## Pertanyaan Refleksi
Jawab dengan kalimat sendiri pada sel di bawah masing-masing pertanyaan.

**1. Mengapa `fol_fc_ask` dijalankan pada KB yang terpisah dari BC?**

*Jawaban:* 

**2. Kapan FC lebih cocok dan kapan BC lebih efisien? Beri contoh dari soal ini.**

*Jawaban:* 

**3. Mengapa KB tidak bisa menyatakan "Sorya melewati batas studi" secara eksplisit? Apa beda "tidak terbukti" dengan "terbukti salah" pada definite clause?**

*Jawaban:* 